In [59]:
import pandas as pd
import time
import random
import requests
from bs4 import BeautifulSoup
import re
import json
from tqdm import tqdm
from unidecode import unidecode 
from datetime import datetime

In [60]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/87.0.4280.88 Safari/537.36'}

### Functions

In [61]:
def scrape_pincali():
    
    # -----------------------------
    # Consultation date
    # -----------------------------
    consultation_date = datetime.now()
    
    # -----------------------------
    # URL
    # -----------------------------
    url = "https://www.pincali.com/inmuebles/propiedades-en-renta"
    
    
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    
    basic_data = soup.find_all("li", class_="property__component")
    
    # -----------------------------
    # Lists of data
    # -----------------------------
    name, type_, bedrooms, bathrooms, latitude, longitude, street, region, locality, price, surface = [], [], [], [], [], [], [], [], [], [], []
    
    # -----------------------------
    # Loop through each property and extract data
    # -----------------------------
    for elementos in basic_data:
        
        # -------- Price --------
        price_tag = elementos.find("div", class_="price")
        price.append(price_tag.get_text(strip=True) if price_tag else None)
        
        # -------- Listing --------
        name_tag = elementos.find("div", class_="title")
        name.append(name_tag.get_text(strip=True) if name_tag else None)
        
        # -------- Lat / Long --------
        lat = elementos.get("data-lat")
        lng = elementos.get("data-long")
        
        latitude.append(float(lat) if lat else None)
        longitude.append(float(lng) if lng else None)
        
        # -------- Ubicación --------
        location_tag = elementos.find("div", class_="location")
        
        if location_tag:
            location_text = location_tag.get_text(strip=True)
            street.append(location_text)
            
            partes = [p.strip() for p in location_text.split(",")]
            locality.append(partes[-1] if len(partes) > 0 else None)
            region.append(partes[-2] if len(partes) > 1 else None)
        else:
            street.append(None)
            region.append(None)
            locality.append(None)
        
        # -------- Features --------
        rec = None
        bath = None
        sup = None
        
        features = elementos.find("div", class_="features")
        
        if features:
            for item in features.find_all("div"):
                texto = item.get_text(strip=True).lower()
                
                numero = re.search(r"\d+", texto)
                numero = int(numero.group()) if numero else None
                
                if "rec" in texto:
                    rec = numero
                elif "bañ" in texto:
                    bath = numero
                elif "m²" in texto or "m2" in texto:
                    sup = numero
        
        bedrooms.append(rec)
        bathrooms.append(bath)
        surface.append(sup)
        
        type_.append(None)
    
    # -----------------------------
    # Final DataFrame
    # -----------------------------
    table = pd.DataFrame({
        "name": name,
        "price": price,
        "type": type_,
        "surface": surface,
        "bedrooms": bedrooms,
        "bathrooms": bathrooms,
        "latitude": latitude,
        "longitude": longitude,
        "street": street,
        "region": region,
        "locality": locality,
        "consultation_date": consultation_date
    })

    #Country
    country = "Mexico"
    table["country"] = country
    table["source"] = "Pincali"
    
    return table

In [ ]:
def icasas(estado, tipo="venta"):
    if tipo== "venta":
        base_url = "https://www.icasas.mx/venta/habitacionales-departamentos-{}-2_3_1_0_0_0/list/p_{}"
    elif tipo== "renta":
        base_url = "https://www.icasas.mx/renta/habitacionales-departamentos-{}-2_3_1_0_0_0/list/p_{}"
    else:
        raise ValueError("Tipo de propiedad no válido. Usa 'venta' o 'renta'.")
    all_data=pd.DataFrame()
    for paginas in tqdm(range(1, 5)):
        url= base_url.format(estado, paginas)
        r=requests.get(url, headers=headers)
        soup=BeautifulSoup(r.text, 'html.parser')
        resultados=soup.find_all('li', class_='serp-snippet ad featured')
        oferta, precio, superficie, recamaras, bathrooms, lat, lon = [], [], [], [], [], [], []
        for i in resultados:
            a_tag = i.find('a', class_='detail-redirection')
            oferta.append(a_tag.get_text(strip=True) if a_tag else None)
            superficie.append(i.find('span', class_='areaBuilt').get_text(strip=True) if i.find('span', class_='areaBuilt') else None)
            recamaras.append(i.find('span', class_='rooms').get_text(strip=True) if i.find('span', class_='rooms') else None)
            bathrooms.append(i.find('span', class_='bathrooms').get_text(strip=True) if i.find('span', class_='bathrooms') else None)
            lat_tag = i.find('meta', itemprop='latitude')
            lon_tag = i.find('meta', itemprop='longitude')
            lat.append(lat_tag['content'] if lat_tag else None)
            lon.append(lon_tag['content'] if lon_tag else None)
            precio.append(i.find('div', class_='price').get_text(strip=True) if i.find('div', class_='price') else None)


        
        temp = pd.DataFrame({'name': oferta, 'price':precio, 'surface': superficie, 'bedrooms': recamaras, 'bathrooms': bathrooms, 'latitude': lat, 'longitude': lon})
        all_data = pd.concat([all_data, temp], ignore_index=True)
    all_data["consultation_date"] = pd.to_datetime("today")
    all_data["source"] = "icasas"
    all_data["country"] = "Mexico"

    return all_data


In [71]:
def scrape_yapo(tipo="venta"):
    data = []
    source_name = "yapo.cl"
    # Consultation date
    consultation_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    for i in tqdm(range(1, 3)):
        url = f"https://www.yapo.cl/searchresult/bienes-raices-{tipo}-apartamentos?o={i}"
        
        try:
            response = requests.get(url, headers=headers, timeout=10)
            
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'html.parser')
                ads = soup.find_all('div', class_='d3-ad-tile__content') 
                
                for ad in ads:
                    details = ad.find_all('li', class_='d3-ad-tile__details-item')
                    
                    bedrooms = None
                    bathrooms = None
                    #Details: Bathrooms and Bedrooms
                    for li in details:
                        icon = li.find('use')
                        if icon:
                            href = icon.get('xlink:href', '')
                            if '#bed' in href:
                                bedrooms = li.get_text(strip=True)
                            elif '#bath' in href:
                                bathrooms = li.get_text(strip=True)

                    item = {
                        "consultation_date": consultation_date,
                        "source": source_name,
                        "locality": ad.find('div', class_='d3-ad-tile__location').get_text(strip=True) if ad.find('div', class_='d3-ad-tile__location') else None,
                        "country": "Chile",
                        "name": ad.find('span', class_='d3-ad-tile__title').get_text(strip=True) if ad.find('span', class_='d3-ad-tile__title') else None,
                        "price": ad.find('div', class_='d3-ad-tile__price').get_text(strip=True) if ad.find('div', class_='d3-ad-tile__price') else None,
                        "bedrooms": bedrooms,
                        "bathrooms": bathrooms
                    }
                    data.append(item)
            
            time.sleep(random.uniform(1, 3))
            
        except Exception as e:
            print(f"Error en página {i}: {e}")
            break

    return pd.DataFrame(data)


In [ ]:
def scrape_properati(domain: str):
    """
    Scrape Properati listings by country domain.

    Parameters
    ----------
    domain : str
        Country domain: 'ar', 'ec', 'co', 'pe'

    Returns
    -------
    pd.DataFrame
    """

    # =========================
    # Validate domain
    # =========================
    allowed_domains = ["ar", "ec", "co", "pe"]
    if domain not in allowed_domains:
        raise ValueError(f"Domain must be one of {allowed_domains}")

    # -----------------------------
    # Country mapping
    # -----------------------------
    country_map = {
        "ar": "Argentina",
        "ec": "Ecuador",
        "co": "Colombia",
        "pe": "Perú"
    }

    country = country_map[domain]

    # =========================
    # Automatic consultation date
    # =========================
    consultation_date = datetime.today().strftime("%Y-%m-%d")

    # =========================
    # URL
    # =========================
    url = f"https://www.properati.com.{domain}/s/alquiler"

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    }

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        raise ConnectionError(f"Request failed with status {response.status_code}")

    soup = BeautifulSoup(response.text, "html.parser")

    # =========================
    # BASIC INFO
    # =========================
    basics = soup.find_all("div", class_="snippet__content")

    price, surface = [], []

    for element in basics:
        price.append(
            element.find("div", class_="price").text.strip()
            if element.find("div", class_="price") else None
        )
        surface.append(
            element.find("span", class_="properties__area").text.strip()
            if element.find("span", class_="properties__area") else None
        )

    basics_df = pd.DataFrame({"price": price, "surface": surface})

    if len(basics_df) > 2:
        basics_df = basics_df.drop([0, 1]).reset_index(drop=True)

    # =========================
    # JSON-LD INFO
    # =========================
    scripts = soup.find_all("script", type="application/ld+json")

    if not scripts:
        raise ValueError("No JSON-LD data found")

    data_json = json.loads(scripts[0].text)[0]['about']

    name, type_, bedrooms, bathrooms = [], [], [], []
    latitude, longitude, street, region, locality = [], [], [], [], []

    for item in data_json:

        type_.append(item.get('@type'))
        bedrooms.append(item.get('numberOfBedrooms'))
        bathrooms.append(item.get('numberOfBathroomsTotal'))

        geo = item.get('geo', {})
        latitude.append(geo.get('latitude'))
        longitude.append(geo.get('longitude'))

        address = item.get('address', {})
        street.append(address.get('streetAddress'))
        region.append(address.get('addressRegion'))
        locality.append(address.get('addressLocality'))

        name.append(item.get('name'))

    table = pd.DataFrame({
        'name': name,
        'type': type_,
        'street': street,
        'region': region,
        'locality': locality,
        'bedrooms': bedrooms,
        'bathrooms': bathrooms,
        'latitude': latitude,
        'longitude': longitude
    })

    # =========================
    # Merge
    # =========================
    final_df = pd.concat([table, basics_df], axis=1)

    # Add automatic consultation date
    final_df["consultation_date"] = consultation_date
    final_df["country"] = country
    final_df["source"] = "Properati"
    return final_df

## Extracción de información

#### Argentina, Ecuador, Peru, Colombia

In [65]:
domains = ["ar", "ec", "co", "pe"]
#Loop through domains and concatenate results
final_results = pd.DataFrame()
for domain in tqdm(domains):
    try:
        df = scrape_properati(domain)
        final_results = pd.concat([final_results, df], ignore_index=True)
    except Exception as e:
        print(f"Error scraping domain {domain}: {e}")
final_results

100%|██████████| 4/4 [00:04<00:00,  1.19s/it]


,name,type,street,region,locality,bedrooms,bathrooms,latitude,longitude,price,surface,consultation_date,country,source
0,Casa en Alquiler en Villa Gesell,House,"Paseo 108 502-600, Villa Gesell, B7165, Buenos...",Buenos Aires Costa Atlántica,Villa Gesell,2.0,2.0,-37.258423,-56.974463,$ 150.000,None,2026-03-06,Argentina,Properati
1,Departamento en Alquiler en Villa Gesell,Apartment,"Paseo Ciento Veinticinco 125, Villa Gesell, B7...",Buenos Aires Costa Atlántica,Villa Gesell,2.0,2.0,-37.276382,-56.98152,$ 50.000,45 m²,2026-03-06,Argentina,Properati
2,Departamento en Alquiler en Villa Gesell,Apartment,"Mis Ilusiones, Villa Gesell, B7165, Provincia ...",Buenos Aires Costa Atlántica,Villa Gesell,2.0,2.0,-37.274216,-56.98106,$ 60.000,60 m²,2026-03-06,Argentina,Properati
3,Departamento en Alquiler en Villa Gesell,Apartment,"San Jorge Ii, Villa Gesell, Provincia de Bueno...",Buenos Aires Costa Atlántica,Villa Gesell,1.0,1.0,-37.274216,-56.98106,$ 60.000,45 m²,2026-03-06,Argentina,Properati
4,Departamento en Alquiler en Villa Gesell,Apartment,"Edificio Malena, Villa Gesell, Provincia de Bu...",Buenos Aires Costa Atlántica,Villa Gesell,1.0,1.0,-37.293316,-56.995737,$ 60.000,50 m²,2026-03-06,Argentina,Properati
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,Local Comercial en Alquiler en San Juan de Lur...,Accommodation,"Av. Los Duraznos 423, Lima, Perú",Lima,Lima,NaN,3.0,-11.9814359,-77.0046887,S/.31,"1,355 m²",2026-03-06,Perú,Properati
116,Almacén en Alquiler en Chilca,Accommodation,"Chilca, Av. Huancavelica, Chilca, Perú",Junín,Huancayo,NaN,NaN,-12.0865194,-75.2079297,"USD3,900",None,2026-03-06,Perú,Properati
117,Oficina en Alquiler en Surquillo,Accommodation,"Av. El Sauce 100, Surquillo, Perú",Lima,Lima,NaN,1.0,-12.1239516,-77.0007231,"S/.1,100",20 m²,2026-03-06,Perú,Properati
118,Local Comercial en Alquiler en San Isidro,Accommodation,"La Putanesca, Avenida Los Conquistadores 137, ...",Lima,Lima,NaN,NaN,-12.09716792,-77.03564643,"USD12,000",500 m²,2026-03-06,Perú,Properati


#### Mexico

In [66]:
pincali_mx= scrape_pincali()

In [67]:
icasas_mx= icasas("distrito-federal","renta")

Scrapeando icasas en distrito-federal: 100%|██████████| 4/4 [00:04<00:00,  1.09s/it]


#### Chile

In [68]:
yapo= scrape_yapo(tipo="alquiler")

In [ ]:
# Concatenate all dataframes
tabla_final=pd.concat([final_results, pincali_mx, icasas_mx, yapo], ignore_index=True)

print("Total de registros:", len(tabla_final))
tabla_final

Total de registros: 318


,name,type,street,region,locality,bedrooms,bathrooms,latitude,longitude,price,surface,consultation_date,country,source
0,Casa en Alquiler en Villa Gesell,House,"Paseo 108 502-600, Villa Gesell, B7165, Buenos...",Buenos Aires Costa Atlántica,Villa Gesell,2.0,2.0,-37.258423,-56.974463,$ 150.000,None,2026-03-06,Argentina,Properati
1,Departamento en Alquiler en Villa Gesell,Apartment,"Paseo Ciento Veinticinco 125, Villa Gesell, B7...",Buenos Aires Costa Atlántica,Villa Gesell,2.0,2.0,-37.276382,-56.98152,$ 50.000,45 m²,2026-03-06,Argentina,Properati
2,Departamento en Alquiler en Villa Gesell,Apartment,"Mis Ilusiones, Villa Gesell, B7165, Provincia ...",Buenos Aires Costa Atlántica,Villa Gesell,2.0,2.0,-37.274216,-56.98106,$ 60.000,60 m²,2026-03-06,Argentina,Properati
3,Departamento en Alquiler en Villa Gesell,Apartment,"San Jorge Ii, Villa Gesell, Provincia de Bueno...",Buenos Aires Costa Atlántica,Villa Gesell,1.0,1.0,-37.274216,-56.98106,$ 60.000,45 m²,2026-03-06,Argentina,Properati
4,Departamento en Alquiler en Villa Gesell,Apartment,"Edificio Malena, Villa Gesell, Provincia de Bu...",Buenos Aires Costa Atlántica,Villa Gesell,1.0,1.0,-37.293316,-56.995737,$ 60.000,50 m²,2026-03-06,Argentina,Properati
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
313,Depto tipo Estudio / La Cisterna,NaN,NaN,NaN,La Cisterna,None,1,NaN,NaN,$250.000-4%Oportunidad,NaN,2026-03-06 01:07:42,Chile,yapo.cl
314,Nueva Andrés Bello 1826 Depto 2 Dorm con Terra...,NaN,NaN,NaN,Independencia,2,1,NaN,NaN,$346.000-1%,NaN,2026-03-06 01:07:42,Chile,yapo.cl
315,"Arriendo, DEPARTAMENTO, Independencia. 1D/1B",NaN,NaN,NaN,Independencia,1,1,NaN,NaN,$230.000,NaN,2026-03-06 01:07:42,Chile,yapo.cl
316,Mapocho 3549 Depto 1 Dorm con Terraza 2 MESES ...,NaN,NaN,NaN,Quinta Normal,1,1,NaN,NaN,$292.955-4%,NaN,2026-03-06 01:07:42,Chile,yapo.cl


In [70]:
### Save to CSV
tabla_final.to_csv("rent_lac_data_example.csv", index=False, encoding="utf-8-sig")